In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgbm

In [3]:
# 데이터 분할 및 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 특성 선택
from sklearn.feature_selection import SequentialFeatureSelector

# 하이퍼파라미터 튜닝
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_recall_curve
# 모델 평가 지표
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# 데이터 시각화
import seaborn as sns

### 1. 데이터 로드 및 탐색

#### 1.1 데이터 로드

In [4]:
df_trn = pd.read_csv('train.csv')
df_tst = pd.read_csv('test.csv')

### 2. 데이터 전처리

#### 2.2 파생 변수 생성

In [5]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 계절을 반환하는 함수는 그대로 사용
def get_season(month):
    if 3 <= month <= 5:
        return 'spring'
    elif 6 <= month <= 8:
        return 'summer'
    elif 9 <= month <= 11:
        return 'autumn'
    else:
        return 'winter'

def preprocess_data(df_trn, df_tst):
    # pdays: -1 → 999
    df_trn['pdays'] = df_trn['pdays'].replace(-1, 999)
    df_tst['pdays'] = df_tst['pdays'].replace(-1, 999)

    # log 변환: balance / duration
    df_trn['balance_log']  = np.log1p(df_trn['balance'])
    df_tst['balance_log']  = np.log1p(df_tst['balance'])
    df_tst['duration_log']  = np.log1p(df_tst['duration'])
    df_tst['duration_log']  = np.log1p(df_tst['duration'])

    # duration per campaign 변수 생성
    df_trn['duration_per_campaign'] = df_trn['duration'] / (df_trn['campaign'] + 1)
    df_tst['duration_per_campaign'] = df_tst['duration'] / (df_tst['campaign'] + 1)

    #  day_of_year
    df_trn['day_of_year'] = df_trn['day'] + (df_trn['month'] - 1) * 30
    df_tst['day_of_year'] = df_tst['day'] + (df_tst['month'] - 1) * 30
    
    # poutcome == success 일때, 1이고, 나머지 poutcome 관련된 것은 0으로 반환할 수 있게.
    df_trn['poutcome_success'] = (df_trn['poutcome'] == 'success').astype(int)
    df_tst['poutcome_success'] = (df_tst['poutcome'] == 'success').astype(int)
    
    return df_trn, df_tst

# 범주형 라벨 인코딩
cat_cols = ['job','marital','contact','poutcome','education','default','housing','loan','month']
label_encoder = {}
for col in cat_cols:
    le = LabelEncoder()
    df_trn[col] = le.fit_transform(df_trn[col])
    df_tst[col] = le.transform(df_tst[col])
    label_encoder[col] = le




In [6]:
# 전처리 함수 적용 후 할당
df_trn_pre, df_tst_pre = preprocess_data(df_trn, df_tst)

#### 2.3 Feature와 Target 분리

In [11]:
total_cols = [x for x in df_trn.columns]

id_cols = ['ID']
y_col = ['label']
du_col = ['duration']
x_cols = [x for x in total_cols if x not in id_cols+y_cols]

In [12]:
df_trn_x = df_trn_pre[x_cols]
df_trn_y = df_trn_pre[y_col].values.ravel()
df_tst_x = df_tst_pre[x_cols]

In [19]:
X_train, X_valid, y_train, y_valid = train_test_split(
    df_trn_x, df_trn_y, test_size=0.2, stratify=df_trn_y, random_state=42)

In [20]:
from sklearn.preprocessing import PowerTransformer, LabelEncoder
from sklearn.compose      import ColumnTransformer
from sklearn.pipeline     import Pipeline
from lightgbm             import LGBMClassifier

In [21]:
# duration 만 power transformer 적용, 나머지는 pass through
# preprocessor로 duration 변환과 lgbm 분류기 적용

In [22]:
x_cols = [c for c in X_train.columns if c not in du_col]

preprocessor = ColumnTransformer([
    ('pt_duration', PowerTransformer(), du_col),
    ('passthrough', 'passthrough', x_cols),
], verbose_feature_names_out=False)

pipe = Pipeline([
    ('pre', preprocessor),
    ('clf',  lgbm.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'))
])

In [23]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('pt_duration',
                                                  PowerTransformer(),
                                                  ['duration']),
                                                 ('passthrough', 'passthrough',
                                                  ['age', 'job', 'marital',
                                                   'education', 'default',
                                                   'balance', 'housing', 'loan',
                                                   'contact', 'day', 'month',
                                                   'campaign', 'pdays',
                                                   'previous', 'poutcome',
                                                   'balance_log',
                                                   'duration_per_campaign',
                                                   'day_of_year',
                                                   'poutcome_success'])],
                                   verbose_feature_names_out=False)),
                ('clf',
                 LGBMClassifier(class_weight='balanced', random_state=42,
                                verbose=-1))])

In [24]:
# F1-score: 0.5987856141989725

#### 3.4 변수 선택법

In [25]:
full_pre = pipe.named_steps['pre']

In [26]:
X_train_pre = full_pre.fit_transform(X_train)
X_valid_pre = full_pre.transform(X_valid)

In [27]:
sfs = SequentialFeatureSelector(
    estimator=lgbm.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'),
    n_features_to_select='auto', 
    direction='forward', # 전진선택
    scoring='f1',
    cv=5,
    n_jobs=-1
)

In [28]:
sfs.fit(X_train_pre, y_train)

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted w

SequentialFeatureSelector(estimator=LGBMClassifier(class_weight='balanced',
                                                   random_state=42,
                                                   verbose=-1),
                          n_jobs=-1, scoring='f1')

In [29]:
feature_names = full_pre.get_feature_names_out()
selected = feature_names[sfs.get_support()]
#selected = ['age' ,'housing', 'contact', 'day' ,'month' ,'pdays', 'poutcome' ,'duration_log', 'day_of_year' ,'poutcome_success']
print("Selected features:", selected)

Selected features: ['duration' 'age' 'default' 'housing' 'contact' 'day' 'pdays' 'poutcome'
 'day_of_year' 'poutcome_success']


In [30]:
X_pre_array = full_pre.transform(X_train)  
feature_names = full_pre.get_feature_names_out()

X_pre_df = pd.DataFrame(
    X_pre_array,
    columns=feature_names,
    index=X_train.index  # 원본 인덱스를 유지하고 싶다면
)

# SFS로 선택된 변수 리스트
selected = feature_names[sfs.get_support()]

# 선택된 컬럼만 뽑아서 새 DataFrame 생성
X_train_selected = X_pre_df[selected]
X_train_selected.head()

,duration,age,default,housing,contact,day,pdays,poutcome,day_of_year,poutcome_success
17675,-2.051312,42.0,0.0,1.0,0.0,9.0,252.0,1.0,69.0,0.0
31014,2.034470,41.0,0.0,1.0,0.0,13.0,293.0,2.0,223.0,0.0
7387,-1.088248,44.0,0.0,1.0,0.0,30.0,999.0,3.0,120.0,0.0
26954,-0.109252,51.0,0.0,1.0,0.0,22.0,999.0,3.0,232.0,0.0
16541,-0.440875,53.0,0.0,0.0,0.0,29.0,999.0,3.0,149.0,0.0


In [31]:
X_valid_df = pd.DataFrame(
    full_pre.transform(X_valid),
    columns=feature_names,
    index=X_valid.index
)
X_valid_selected = X_valid_df[selected]

In [32]:
# ['age' 'housing' 'contact' 'day' 'month' 'pdays' 'poutcome' 'duration_log','day_of_year' 'poutcome_success']

#### 선택된 변수 할당하기

### 4. 모델 최적화

#### 4.1 하이퍼파라미터 튜닝

In [33]:
def objective(trial):
    params = {
        'class_weight': 'balanced',
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1.0),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
    }
    clf = LGBMClassifier(**params, random_state=42, verbose=-1)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X_train_selected, y_train, scoring='f1', cv=cv, n_jobs=-1)
    return scores.mean()

#### 4.2 Optuna

##### LGBM Optuna 

In [34]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold


In [35]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)

[I 2025-06-01 01:31:56,770] A new study created in memory with name: no-name-a0fa1865-7bfb-4e35-a2a0-bab9d6ab81fe
[I 2025-06-01 01:31:58,977] Trial 0 finished with value: 0.602391600649714 and parameters: {'learning_rate': 0.26104661870390516, 'n_estimators': 329, 'max_depth': 4, 'subsample': 0.7208376163691159, 'colsample_bytree': 0.9208838385618612, 'min_child_samples': 62}. Best is trial 0 with value: 0.602391600649714.
[I 2025-06-01 01:32:12,604] Trial 1 finished with value: 0.5757373608261684 and parameters: {'learning_rate': 0.005900896577238538, 'n_estimators': 743, 'max_depth': 8, 'subsample': 0.5329608939262286, 'colsample_bytree': 0.7473596947500217, 'min_child_samples': 127}. Best is trial 0 with value: 0.602391600649714.
[I 2025-06-01 01:32:29,838] Trial 2 finished with value: 0.5575425740992859 and parameters: {'learning_rate': 0.0014396102133292722, 'n_estimators': 802, 'max_depth': 10, 'subsample': 0.5374998950981524, 'colsample_bytree': 0.5500646626821897, 'min_child_sa

Best params: {'learning_rate': 0.06817151865983032, 'n_estimators': 608, 'max_depth': 7, 'subsample': 0.9984092306999814, 'colsample_bytree': 0.8375094339139457, 'min_child_samples': 92}


In [36]:
from sklearn.metrics import f1_score, precision_recall_curve

# 최적 파라미터로 모델 학습
best_clf = LGBMClassifier(**study.best_params, random_state=42, verbose=-1)
best_clf.fit(X_train_selected, y_train)

# 검증세트 예측 확률
probs = best_clf.predict_proba(X_valid_selected)[:,1]

# Precision-Recall 커브에서 threshold 찾기
precision, recall, thresholds = precision_recall_curve(y_valid, probs)
f1s = 2 * precision * recall / (precision + recall + 1e-9)
best_idx = f1s.argmax()
best_thr = thresholds[best_idx]
print(f"Best threshold: {best_thr:.4f}, F1: {f1s[best_idx]:.4f}")


Best threshold: 0.1986, F1: 0.6244


In [37]:
#Best threshold: 0.2459, F1: 0.6114

#[feature 선택]
#Best threshold: 0.1938, F1: 0.6259


In [38]:
import optuna
from sklearn.experimental            import enable_hist_gradient_boosting  
from sklearn.ensemble                import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# StratifiedKFold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# HGB(hist gradient boosting) Optuna Tuning 
def objective_hgb(trial):
    params = {
        'learning_rate':   trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
        'max_iter':        trial.suggest_int('max_iter', 100, 500),
        'max_depth':       trial.suggest_int('max_depth', 3, 10),
        'min_samples_leaf':trial.suggest_int('min_samples_leaf', 20, 100),
        'l2_regularization': trial.suggest_loguniform('l2_regularization', 1e-4, 10),
        'max_bins':        trial.suggest_int('max_bins', 100, 255),
        'random_state':    42
    }
    clf = HistGradientBoostingClassifier(**params)
    return cross_val_score(clf, X_train_selected, y_train, scoring='f1', cv=cv, n_jobs=1).mean()



# CatBoost Optuna Tuning 
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 800),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-2, 10),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_uniform('colsample_bylevel', 0.5, 1.0),
        'auto_class_weights': 'Balanced',
        'thread_count':      1,
        'eval_metric': 'F1',
        'random_state': 42,
        'verbose': 0
    }
    clf = CatBoostClassifier(**params)
    score = cross_val_score(clf, X_train_selected, y_train, scoring='f1', cv=cv).mean()
    return score



##### HGB optuna optimize

In [39]:
study_hgb = optuna.create_study(direction='maximize')
study_hgb.optimize(objective_hgb, n_trials=30, n_jobs=1)

[I 2025-06-01 01:39:18,555] A new study created in memory with name: no-name-005508b8-639b-47bb-a808-36f3a7632a74
[I 2025-06-01 01:39:20,298] Trial 0 finished with value: 0.22152360058250542 and parameters: {'learning_rate': 0.015068887975059111, 'max_iter': 152, 'max_depth': 3, 'min_samples_leaf': 85, 'l2_regularization': 1.1179429298880834, 'max_bins': 104}. Best is trial 0 with value: 0.22152360058250542.
[I 2025-06-01 01:39:21,888] Trial 1 finished with value: 0.5241464490382693 and parameters: {'learning_rate': 0.15962440269250844, 'max_iter': 337, 'max_depth': 4, 'min_samples_leaf': 77, 'l2_regularization': 0.006046392330185556, 'max_bins': 195}. Best is trial 1 with value: 0.5241464490382693.
[I 2025-06-01 01:39:24,240] Trial 2 finished with value: 0.5261961341980556 and parameters: {'learning_rate': 0.12996939007941, 'max_iter': 420, 'max_depth': 4, 'min_samples_leaf': 82, 'l2_regularization': 0.06715398481390376, 'max_bins': 157}. Best is trial 2 with value: 0.5261961341980556

In [40]:
print("HGB Best Params:", study_hgb.best_params)
print("HGB Best CV F1:", study_hgb.best_value)

HGB Best Params: {'learning_rate': 0.06985586277685692, 'max_iter': 283, 'max_depth': 10, 'min_samples_leaf': 64, 'l2_regularization': 0.005157493207860865, 'max_bins': 170}
HGB Best CV F1: 0.5417046594707159


In [41]:
#HGB Best Params: {'learning_rate': 0.16118701671509253, 'max_iter': 222, 'max_depth': 9, 'min_samples_leaf': 45, 'l2_regularization': 9.232385210488296, 'max_bins': 252}
# HGB Best CV F1: 0.5502500636269988

##### CatBoost optuna optimize

In [42]:
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=30, n_jobs=1)


[I 2025-06-01 01:43:05,094] A new study created in memory with name: no-name-0b2f9cdf-8088-4b19-92e8-ed8c9c83f2ec
[I 2025-06-01 01:43:07,130] Trial 0 finished with value: 0.5622737147627705 and parameters: {'iterations': 160, 'learning_rate': 0.03172101801209723, 'depth': 9, 'l2_leaf_reg': 0.05723085946789434, 'subsample': 0.8128030110043424, 'colsample_bylevel': 0.9502021504789051}. Best is trial 0 with value: 0.5622737147627705.
[I 2025-06-01 01:43:09,420] Trial 1 finished with value: 0.5960678335015405 and parameters: {'iterations': 317, 'learning_rate': 0.26176235312510643, 'depth': 4, 'l2_leaf_reg': 1.5697236922209556, 'subsample': 0.5020049017031292, 'colsample_bylevel': 0.7650185360379855}. Best is trial 1 with value: 0.5960678335015405.
[I 2025-06-01 01:43:15,223] Trial 2 finished with value: 0.528952387970094 and parameters: {'iterations': 271, 'learning_rate': 0.004014129282914797, 'depth': 10, 'l2_leaf_reg': 0.320370005619202, 'subsample': 0.5862309700073998, 'colsample_byle

In [43]:
print("CatBoost Best Params:", study_cat.best_params)
print("CatBoost Best CV F1:", study_cat.best_value)

CatBoost Best Params: {'iterations': 617, 'learning_rate': 0.060338714960022154, 'depth': 7, 'l2_leaf_reg': 1.2656331133037133, 'subsample': 0.64359967973576, 'colsample_bylevel': 0.735413388366557}
CatBoost Best CV F1: 0.6059367314727988


In [44]:
#CatBoost Best Params: {'iterations': 645, 'learning_rate': 0.04128090321065735, 'depth': 7, 'l2_leaf_reg': 0.6765943053819811, 'subsample': 0.6093501014647345, 'colsample_bylevel': 0.6773569261342616}
#CatBoost Best CV F1: 0.6066541601853407

In [45]:
# 이 과정은 stacking 과정이 제대로 진행되지 않아 ChatGPT 도움으로 코드 작성 진행했습니다.

In [46]:
from sklearn.pipeline       import Pipeline
from sklearn.compose        import ColumnTransformer
from sklearn.preprocessing  import FunctionTransformer, OneHotEncoder, RobustScaler, PowerTransformer
from sklearn.impute         import SimpleImputer
from sklearn.ensemble       import StackingClassifier, RandomForestClassifier
from sklearn.linear_model   import LogisticRegression
from lightgbm               import LGBMClassifier
from catboost               import CatBoostClassifier
import numpy as np

# SFS로 이미 뽑아둔 선택된 변수 목록 저의
selected_features = [
    'duration', 'age', 'default', 'housing', 'contact',
    'day', 'pdays', 'poutcome', 'day_of_year', 'poutcome_success'
]

# 컬럼그룹 정의
numeric_passthrough = ['age', 'day_of_year']
small_scale        = ['day']
large_scale        = ['pdays']
log_trans          = ['duration'] # 로그변환
cat_ohe            = ['default', 'housing', 'contact', 'poutcome', 'poutcome_success']

# 변수에서 결측치 관련 error, stacking 할 데이터의 값이 너무 크다는 error가 발생해 직접적으로 전처리 과정을 진행할 수 있는 transformer 정의

full_pre_sel = ColumnTransformer([
    # 숫자 그대로 통과시키고 결측 채움
    ('num_imp',    SimpleImputer(strategy='median'), numeric_passthrough),

    # 작게 분포된 수치형 → RobustScaler
    ('small_scale', Pipeline([
         ('imp',   SimpleImputer(strategy='median')),
         ('scale', RobustScaler())
    ]), small_scale),

    # 크게 분포된 수치형 → RobustScaler
    ('large_scale', Pipeline([
         ('imp',   SimpleImputer(strategy='median')),
         ('scale', RobustScaler())
    ]), large_scale),

    # 로그 변환이 유용한 수치형
    ('log_pt', Pipeline([
         ('imp', SimpleImputer(strategy='median')),
         ('pt',  PowerTransformer(method='yeo-johnson'))
    ]), log_trans),

    # (5) 범주형 → 최빈값 대체 + one-hot
    ('cat', Pipeline([
         ('imp', SimpleImputer(strategy='most_frequent')),
         ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_ohe),
], verbose_feature_names_out=True)

# PipeLine 구축
preprocessor_sel = Pipeline([
    # selected_features 컬럼만 뽑기
    ('selector', FunctionTransformer(lambda df: df[selected_features], validate=False)),
    # 전처리기 적용
    ('full_pre_sel', full_pre_sel),
    #  Inf/NaN → 0으로 마무리 (결측치에 대한 처리)
    ('fix_inf', FunctionTransformer(
        lambda X: np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0),
        validate=False
    ))
])

# Base 모델들 - HGB의 경우 관련된 모델의 f1 스코어가 낮아 제외함.
lgbm_clf = LGBMClassifier(**lgbm_params, random_state=42)
cat_clf  = CatBoostClassifier(**cat_params, random_state=42, verbose=0)

# 스태킹 정의
stack = StackingClassifier(
    estimators=[
        ('lgbm', lgbm_clf),
        ('cat',  cat_clf),
    ],
    final_estimator=LogisticRegression( #LR 로 estimator 정의.
        class_weight='balanced', 
        max_iter=1000,
        random_state=42
    ),
    cv=5,
    n_jobs=-1,
    passthrough=True
)

# 최종 stacking 파이프라인 조립
stack_pipe = Pipeline([
    # selected_features 뽑고 → 전처리
    ('select_pre', preprocessor_sel),
    # 스태킹
    ('stack',      stack),
])

# 학습
stack_pipe.fit(X_train, y_train)


NameError: name 'lgbm_params' is not defined

In [ ]:
from sklearn.metrics import f1_score, classification_report, precision_recall_curve
import numpy as np

# 1) 검증 세트에 대한 클래스 예측
y_valid_pred = stack_pipe.predict(X_valid)

# 2) 기본 F1 score (threshold=0.5 기준)
f1 = f1_score(y_valid, y_valid_pred)
print(f"기본 F1 score (threshold=0.5): {f1:.4f}")

# 3) 자세한 리포트 출력
print(classification_report(y_valid, y_valid_pred))



기본 F1 score (threshold=0.5): 0.6114
              precision    recall  f1-score   support

           0       0.98      0.88      0.93      5590
           1       0.48      0.84      0.61       740

    accuracy                           0.88      6330
   macro avg       0.73      0.86      0.77      6330
weighted avg       0.92      0.88      0.89      6330



In [ ]:
#  검증 셋에서 최적 threshold 재계산
y_valid_proba = stack_pipe.predict_proba(X_valid)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_valid, y_valid_proba)

f1_by_thresh = [
    f1_score(y_valid, (y_valid_proba > t).astype(int))
    for t in thresholds
]
best_idx    = np.argmax(f1_by_thresh)
best_thresh = thresholds[best_idx]
best_f1     = f1_by_thresh[best_idx]
print(f"최종 모델 최적 F1 (검증 기준): {best_f1:.4f} (threshold = {best_thresh:.3f})")

최종 모델 최적 F1 (검증 기준): 0.7120 (threshold = 0.759)


#### 최종 예측 및 Submission 파일 생성

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_recall_curve


X_train_pre = df_trn_pre[x_cols]
X_test_pre  = df_tst_pre[x_cols]
test_ids    = df_tst_pre['ID'] 

y_test_proba = stack_pipe.predict_proba(df_tst_pre)[:, 1]
y_test_pred  = (y_test_proba > best_thresh).astype(int)

# 4) submission 파일 생성
submission = pd.DataFrame({
    'ID':    test_ids,
    'label': y_test_pred
})
submission.to_csv('submission_final_stack_en_(send).csv', index=False)
print("submission.csv 파일 생성 완료!")


✅ submission.csv 파일 생성 완료!
